# 2_self_promotion_new.ipynb

**Purpose:** Identify which early-career researchers self-promoted their first
publication on Twitter, by comparing author names (from the bibliometric record)
with the names and handles of the Twitter users who tweeted the publication's
DOI. See the manuscript Methods ("Self-promotion on Twitter") for the full
description of this name-matching procedure and its validation.

**Manuscript:** Gender differences in online visibility of early-career researchers

**Required input files:**
- `result/0_author_doi_3y_2012_2016_sample.csv` (from `1_Data_Processing_new.ipynb`):
  `author_id`, `first_name`, `last_name`, `doi`
- `result/1_doi_3y_2012_2016_tweet_sample.pkl` (from `1_Data_Processing_new.ipynb`):
  `doi`, `Original_Tweet_list`, `Original_Tweeter_list`, `tw_name_list`,
  `tw_handle_list`, `searched`
- `data/00_openalex_tweeterid_name_processed.pkl`: OpenAlex author names linked
  to Twitter user IDs (`tweeter_id`, `author_name`); see Mongeon, Bowman &
  Costas (2023), "An open dataset of scholars on Twitter"

**Output:** `result/2_self_promotion_scores_sample.csv` -- name-similarity
scores between each author and the users who tweeted their publication


In [ ]:
import re
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
RESULTS_DIR = Path("result")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ENCODING = "utf-8"
pd.set_option("max_colwidth", 100)
pd.set_option("display.float_format", lambda x: "%.4f" % x)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 15)

SIMILARITY_THRESHOLD_GENERAL = 0.6  # Methods: general threshold (0.8 for Chinese names)

TITLES = [
    "mr ", "mr. ", "dr. ", "dr ", "mrs ", "mrs. ", "prof. ", "prof ",
    "ms ", "ms. ", "phd. ", "phd ",
]


def as_list(x):
    """Return x if it is a list, otherwise an empty list (missing values)."""
    return x if isinstance(x, list) else []

### 1. Read datasets

1. Early-career publications (first three years) with author names
2. Tweet information for these publications

In [ ]:
author_12_16_early_paper = pd.read_csv(RESULTS_DIR / "0_author_doi_3y_2012_2016_sample.csv", encoding=ENCODING)

required = ["author_id", "first_name", "last_name", "doi"]
missing = [c for c in required if c not in author_12_16_early_paper.columns]
if missing:
    raise ValueError(f"0_author_doi_3y_2012_2016_sample.csv is missing {missing}; "
                     "re-run 1_Data_Processing_new.ipynb")

author_12_16_early_paper_tw = pd.read_pickle(RESULTS_DIR / "1_doi_3y_2012_2016_tweet_sample.pkl")
for col in ["Original_Tweet_list", "Original_Tweeter_list", "tw_name_list", "tw_handle_list"]:
    author_12_16_early_paper_tw[col] = author_12_16_early_paper_tw[col].apply(as_list)
author_12_16_early_paper_tw["len_tweet"] = author_12_16_early_paper_tw["Original_Tweet_list"].apply(len)
author_12_16_early_paper_tw["len_tweeter"] = author_12_16_early_paper_tw["Original_Tweeter_list"].apply(len)

# Altmetric results are keyed by DOI only; attach the author ID if the tweet
# data do not already contain it.
if "author_id" not in author_12_16_early_paper_tw.columns:
    author_12_16_early_paper_tw = author_12_16_early_paper_tw.merge(
        author_12_16_early_paper[["author_id", "doi"]].drop_duplicates(), on="doi", how="left"
    )

### 2. Prepare author names and tweeters

- Step 1: clean the author names of publications with tweets
- Step 2: list the tweeters of each publication

In [ ]:
def name_revision(name_set: pd.DataFrame) -> pd.DataFrame:
    """Clean first names: replace hyphens/periods with spaces and keep only
    the first token if it is more than one character."""
    name_set[["last_name", "first_name"]] = name_set[["last_name", "first_name"]].fillna("NULL").astype(str)
    name_set["first_name_revised"] = (
        name_set["first_name"]
        .apply(lambda x: x.replace("-", " ").replace(".", " "))
        .apply(lambda x: re.split(" ", x)[0] if (" " in x) and (len(re.split(" ", x)[0]) > 1) else x)
    )
    return name_set


# One name per author (first non-missing value)
author_12_16_name = (
    author_12_16_early_paper[["author_id", "first_name", "last_name"]]
    .groupby("author_id")
    .agg({"last_name": list, "first_name": list})
    .reset_index()
)
for col in ["last_name", "first_name"]:
    author_12_16_name[col] = author_12_16_name[col].apply(
        lambda x: next((y for y in dict.fromkeys(x) if pd.notna(y)), "@")
    )

# Publications with at least one tweet
author_12_16_tw_noNan = author_12_16_early_paper_tw[author_12_16_early_paper_tw["len_tweet"] > 0].copy()
author_12_16_tw_noNan = author_12_16_tw_noNan.merge(author_12_16_name, on="author_id", how="left")
author_12_16_tw_noNan = author_12_16_tw_noNan.drop_duplicates(["author_id", "doi"])
author_12_16_tw_noNan = name_revision(author_12_16_tw_noNan)

# One row per publication x tweeter
author_tweeter = (
    author_12_16_early_paper_tw.loc[author_12_16_early_paper_tw["len_tweeter"] > 0,
                                    ["author_id", "doi", "Original_Tweeter_list"]]
    .explode("Original_Tweeter_list")
    .rename(columns={"Original_Tweeter_list": "Original_Tweeter_int"})
    .dropna(subset=["Original_Tweeter_int"])
)
print(f"{len(author_12_16_tw_noNan)} publications with tweets; {len(author_tweeter)} publication-tweeter pairs")

### 3. Name-similarity functions

Compare author names with tweeter names. More details are given in the
Methods section ("Self-promotion on Twitter").

In [ ]:
def process_str(x: str) -> str:
    """Lowercase and strip common titles/honorifics before comparison."""
    x = x.replace("-", " ").lower()
    for title in TITLES:
        x = x.replace(title, "")
    return x


def similar(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def compare_multi(f, l, fl, lf, f_l, l_f, candidate):
    """Compare a candidate name with several author-name variants (first, last,
    first+last, last+first, with/without space); return the best similarity."""
    candidate = process_str(candidate)
    if " " in candidate:
        return max(similar(f_l, candidate), similar(l_f, candidate))
    elif candidate != "null":
        return max(similar(f, candidate), similar(l, candidate), similar(fl, candidate), similar(lf, candidate))
    return 0.0


def self_verify(first, last, list1, list2, list3, listT, key):
    """Compare an author's name with three candidate-name sources for the
    tweeters of a publication:
      list1 -- OpenAlex author names linked to each tweeter
      list2 -- Twitter display names of each tweeter
      list3 -- Twitter handles of each tweeter
    Returns the similarity scores, best-matching names, the source of the best
    match, and the ID of the best-matching tweeter."""
    list1, list2, list3, listT = as_list(list1), as_list(list2), as_list(list3), as_list(listT)
    score_1 = score_2 = score_3 = 0.0
    tweeter_id_1, tweeter_id_2, tweeter_id_3 = 1, 2, 3
    author_name = user_name = handle = "NULL"

    first = str(first).replace("@", "")
    last = str(last).replace("@", "")
    first_last, firstlast = f"{first} {last}", f"{first}{last}"
    last_first, lastfirst = f"{last} {first}", f"{last}{first}"

    for idx, names in enumerate(list1):
        for j in as_list(names):
            if isinstance(j, str):
                score = compare_multi(first, last, firstlast, lastfirst, first_last, last_first, j)
                if score > score_1:
                    score_1, author_name = score, j
                    tweeter_id_1 = listT[idx] if idx < len(listT) else 1

    for i, name in enumerate(list2):
        if isinstance(name, str):
            score = compare_multi(first, last, firstlast, lastfirst, first_last, last_first, name)
            if score > score_2:
                score_2, user_name = score, name
                tweeter_id_2 = listT[i] if (key == 0 and i < len(listT)) else 2

    for i, name in enumerate(list3):
        if isinstance(name, str):
            score = compare_multi(first, last, firstlast, lastfirst, first_last, last_first, name)
            if score > score_3:
                score_3, handle = score, name
                tweeter_id_3 = listT[i] if (key == 0 and i < len(listT)) else 3

    max_score = max(score_1, score_2, score_3)
    if max_score == score_1:
        attr, likely_name, tweeter_id = "author_name_self_score", author_name, tweeter_id_1
    elif max_score == score_2:
        attr, likely_name, tweeter_id = "tw_name_self_score", user_name, tweeter_id_2
    else:
        attr, likely_name, tweeter_id = "tw_handle_self_score", handle, tweeter_id_3

    return (score_1, score_2, score_3, author_name, user_name, handle,
            attr, max_score, likely_name, tweeter_id)

### 4. Attach OpenAlex author names to the tweeters

In [ ]:
author_name_openalex = pd.read_pickle(DATA_DIR / "00_openalex_tweeterid_name_processed.pkl")[["tweeter_id", "author_name"]]

# Tweeter IDs are numbers in the OpenAlex file and may be text in the tweet
# data; use text in both so that the merge matches them.
author_name_openalex["tweeter_id"] = author_name_openalex["tweeter_id"].astype(str)
author_tweeter["Original_Tweeter_int"] = author_tweeter["Original_Tweeter_int"].astype(str)

Tweeter_list = author_tweeter.merge(
    author_name_openalex, left_on="Original_Tweeter_int", right_on="tweeter_id", how="left"
)
Tweeter_list["author_name"] = Tweeter_list["author_name"].apply(as_list)
Tweeter_list = Tweeter_list.groupby(["author_id", "doi"], sort=False)["author_name"].apply(list).reset_index()

author_12_16_tw_noNan = author_12_16_tw_noNan.merge(Tweeter_list, on=["author_id", "doi"], how="left")
author_12_16_tw_noNan["author_name"] = author_12_16_tw_noNan["author_name"].apply(as_list)

### 5. Self-promotion detection

In [ ]:
score_cols = [
    "author_name_self_score", "tw_name_self_score", "tw_handle_self_score",
    "likely_author", "likely_name", "likely_handle",
    "max_attribute", "max_score", "most_likely_name", "likely_Tweeter_ID",
]
scores = author_12_16_tw_noNan.apply(
    lambda row: self_verify(
        row["first_name_revised"], row["last_name"], row["author_name"],
        row["tw_name_list"], row["tw_handle_list"], row["Original_Tweeter_list"], row["searched"],
    ),
    axis=1,
    result_type="expand",
)
scores.columns = score_cols
author_12_16_tw_noNan[score_cols] = scores

### 6. Save results and check the match rate

Share of authors with a name-similarity score above the 0.6 threshold.

In [ ]:
self_promotion_cols = [
    "author_id", "doi", "last_name", "first_name", "first_name_revised",
    "len_tweet", "len_tweeter",
    "Original_Tweet_list", "Original_Tweeter_list",
    "tw_name_list", "tw_handle_list", "searched", "author_name",
] + score_cols

author_12_16_tw_noNan[self_promotion_cols].to_csv(
    RESULTS_DIR / "2_self_promotion_scores_sample.csv", index=False, encoding=ENCODING
)

matched_share = (
    author_12_16_tw_noNan.loc[author_12_16_tw_noNan["max_score"] > SIMILARITY_THRESHOLD_GENERAL, "author_id"].nunique()
    / author_12_16_tw_noNan["author_id"].nunique()
)
print(f"Share of authors with a name-match score > {SIMILARITY_THRESHOLD_GENERAL}: {matched_share:.2%}")